# cuda-empty-cache — worked example 2: Gate the release behind cuda.is_available()

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `cuda-empty-cache`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`torch.cuda.empty_cache()` is safe to call on a CPU-only machine — it simply no-ops. A common defensive pattern still gates heavier cleanup with `torch.cuda.is_available()` so you can branch your logic and report what hardware you ran on. The gate reads a boolean; the empty_cache call itself never raises regardless of the gate.

## Worked solution

**Goal.** Run a small accumulation loop, and after it finishes report whether CUDA was available and how many release calls we made.

**Step 1 — read the hardware gate once.** `avail = t.cuda.is_available()` is a cheap boolean. We read it a single time rather than inside the loop, since it cannot change mid-run.

**Step 2 — accumulate.** We sum each tensor's `.sum()` into a running total. This is ordinary tensor work; nothing CUDA-specific.

**Step 3 — release unconditionally inside the loop.** We call `t.cuda.empty_cache()` once per iteration and count it in `n_released`. We deliberately do NOT skip the call when `avail` is False — the whole point is that the call is portable: it no-ops on CPU instead of raising.

**Step 4 — report.** We return a dict with the gate value, the count, and the final accumulated total. On a CPU runner `avail` is False but `n_released` still equals the loop length, demonstrating the call ran every time without error.

In [ ]:
def gated_release(tensors):
    avail = t.cuda.is_available()
    total = t.zeros(())
    n_released = 0
    for x in tensors:
        total = total + x.sum()
        t.cuda.empty_cache()
        n_released += 1
    return {'cuda_available': avail, 'n_released': n_released, 'total': total}

t.manual_seed(0)
tensors = [t.randn(5) for _ in range(4)]
res = gated_release(tensors)
print(res['n_released'], bool(res['cuda_available']))
print(t.allclose(res['total'], t.stack([x.sum() for x in tensors]).sum()))